# 08A Trajectory Preprocessing

This notebook prepares GROMACS production trajectories before RMSD, radius of gyration, SASA, or other structural analysis.

Raw periodic trajectories are often not analysis-ready. A polymer can cross the unit-cell boundary, molecules can look split, and the whole system can drift through the box. Those effects can create noisy RMSD/Rg traces that describe box handling rather than molecular behavior.

## Strategy

The workflow uses a dedicated GROMACS index group named `[ center ]`.

For the current polymer-only workflow, `[ center ]` is generated from `[ PHA ]`. The preprocessing commands then select `center` for centering or fitting and `System` for output.

This keeps the full solvated system, including water and ions, while keeping the polymer centered. The command layer never needs to hardcode `PHA`, so future systems can map `[ center ]` to `Protein`, `MEMB | Protein`, or `Protein | PHA` without rewriting preprocessing calls.

## Why full-group centering

Centering on a single atom is fragile for polymers because one atom can sit near a periodic boundary while the rest of the chain spans another image. Centering on the full target group uses the group geometry, giving a more stable representation for visual inspection and downstream trajectory analysis.

In [ ]:
from pathlib import Path

from iphasimulator.trajectory import ensure_center_index, preprocess_gromacs_trajectory, read_index

SYSTEM_DIR = Path("../examples/output/md_tests/PHB4/gromacs/solvated_polymer").resolve()
SYSTEM_DIR

## Step 1: create or reuse `center.ndx`

If `center.ndx` already contains `[ center ]`, it is reused. Otherwise the helper reads `index.ndx`, resolves the workflow mapping, and writes a reusable `center.ndx` with the original groups plus `[ center ]`.

In [ ]:
center_result = ensure_center_index(
    SYSTEM_DIR / "index.ndx",
    SYSTEM_DIR / "center.ndx",
    workflow_type="polymer",
)

center_index = read_index(center_result.index_path)
{
    "center_index": str(center_result.index_path),
    "created": center_result.created,
    "reused_existing_center": center_result.reused_existing_center,
    "source_groups": center_result.source_groups,
    "center_atom_count": len(center_index.group("center")),
}

## Step 2: center, reconstruct molecules, and compact-wrap

The first `trjconv` pass reconstructs molecules across periodic boundaries, uses a compact unit cell, and centers on `[ center ]`:

```bash
echo -e "center\nSystem" | gmx trjconv \
  -f step7_production.xtc \
  -s step7_production.tpr \
  -n center.ndx \
  -pbc mol \
  -ur compact \
  -center \
  -o processed/step7_centered.xtc
```

Selection 1 is the centering group. Selection 2 is the output group. Use `System` for output unless you intentionally want to discard solvent and ions.

## Step 3: optional fitting/alignment

The second pass removes overall rotation and translation relative to the reference structure:

```bash
echo -e "center\nSystem" | gmx trjconv \
  -f processed/step7_centered.xtc \
  -s step7_production.tpr \
  -n center.ndx \
  -fit rot+trans \
  -o analysis_ready/step7_fitted.xtc
```

The fitted trajectory is the preferred input for RMSD-like analyses. The centered trajectory is useful when fitting is not appropriate for a specific observable.

In [ ]:
required = [SYSTEM_DIR / "step7_production.xtc", SYSTEM_DIR / "step7_production.tpr"]
have_production = all(path.exists() for path in required)

outputs = preprocess_gromacs_trajectory(
    SYSTEM_DIR,
    workflow_type="polymer",
    fit=True,
    extract_representative_frame=True,
    dry_run=not have_production,
)

{
    "have_production": have_production,
    "raw": str(outputs.raw_trajectory_path),
    "centered": str(outputs.centered_trajectory_path),
    "analysis_ready": str(outputs.analysis_trajectory_path),
    "representative_frame": str(outputs.representative_frame_path),
}

If `have_production` is `False`, run the GROMACS production stage first so `step7_production.xtc` and `step7_production.tpr` exist, then re-run the cell above. The notebook still creates `center.ndx` and shows the exact output layout.

## Before and after checks

The raw trajectory is the direct production output. The centered trajectory should show the polymer as one molecule in a compact solvent box. The fitted trajectory should remove whole-system drift and rotation while preserving solvent and ions in the output group.

In [ ]:
for label, path in {
    "raw": outputs.raw_trajectory_path,
    "centered": outputs.centered_trajectory_path,
    "analysis_ready": outputs.analysis_trajectory_path,
    "representative_frame": outputs.representative_frame_path,
}.items():
    if path is not None:
        print(f"{label:20s} {path.exists()}  {path}")

## Visualize representative frames

The preprocessing helper writes `analysis_ready/representative_frame.gro` from the analysis-ready trajectory when production files are available. The cell below uses `nglview` if installed; otherwise it prints the frame path for use in VMD, PyMOL, or GROMACS tools.

In [ ]:
frame_path = outputs.representative_frame_path

if frame_path is None or not frame_path.exists():
    print("Representative frame is not available yet. Generate production trajectory files and rerun preprocessing.")
else:
    try:
        from IPython.display import display
        import nglview as nv
        view = nv.show_file(str(frame_path))
        display(view)
    except ImportError:
        print(frame_path)